# Emotion Classification with Attention & Transformers (Kaggle Pipeline)
This notebook trains multiple models (LSTM, GRU, BiLSTM+Attention, DistilBERT, RoBERTa) on the GoEmotions dataset.
It is optimized for Kaggle T4 x2 GPUs.

## Instructions:
1. Ensure the Accelerator is set to **GPU T4 x2**.
2. Run all cells sequentially.
3. At the end, a zip file will be generated for download containing the models and metrics.

## 1. Environment Setup & Multi-GPU Config

In [ ]:
!pip install -q transformers datasets evaluate accelerate optuna
!pip install -q torchinfo wordcloud

import os
import time
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Multi-GPU check
print(f"PyTorch version: {torch.__version__}")
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs available: {num_gpus}")
if num_gpus > 0:
    for i in range(num_gpus):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Store metrics for all models
project_metrics = []


## 2. Load Dataset & Mapping (27 to 6 Ekman Emotions)

In [ ]:
# Load GoEmotions dataset
dataset = load_dataset("google-research-datasets/go_emotions")

# Define Ekman Mapping
ekman_mapping = {
    'anger': ['anger', 'annoyance', 'disapproval'],
    'disgust': ['disgust'],
    'fear': ['fear', 'nervousness'],
    'joy': ['joy', 'amusement', 'approval', 'excitement', 'gratitude', 'love', 'optimism', 'relief', 'pride', 'admiration', 'desire', 'caring'],
    'sadness': ['sadness', 'disappointment', 'embarrassment', 'grief', 'remorse'],
    'surprise': ['surprise', 'realization', 'confusion', 'curiosity', 'awe'],
    'neutral': ['neutral']
}

# GoEmotions original label names
original_labels = dataset['train'].features['labels'].feature.names

# Create a mapping from original ID to Ekman string
id_to_ekman = {}
for idx, label_str in enumerate(original_labels):
    mapped_ekman = 'neutral' # default
    for ekman_emotion, go_emotions_list in ekman_mapping.items():
        if label_str in go_emotions_list:
            mapped_ekman = ekman_emotion
            break
    id_to_ekman[idx] = mapped_ekman

ekman_labels = sorted(list(ekman_mapping.keys()))
ekman_to_id = {label: idx for idx, label in enumerate(ekman_labels)}

def map_labels(example):
    first_label_id = example['labels'][0]
    ekman_str = id_to_ekman[first_label_id]
    example['ekman_label'] = ekman_to_id[ekman_str]
    return example

# Apply mapping
dataset = dataset.map(map_labels)


## 3. Brief EDA

In [ ]:
train_df = dataset['train'].to_pandas()

plt.figure(figsize=(10, 5))
sns.countplot(data=train_df, x='ekman_label')
plt.title('Distribution of Ekman Emotions in Training Set')
plt.xticks(ticks=range(len(ekman_labels)), labels=ekman_labels)
plt.show()


## 4. Preprocessing for Sequence Models

In [ ]:
import re
import urllib.request
import zipfile
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset

# Compute Class Weights to handle imbalance
labels_for_weights = train_df['ekman_label'].values
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels_for_weights), y=labels_for_weights)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)


def tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

word_counts = Counter()
for text in train_df['text']:
    word_counts.update(tokenize(text))

vocab = {'<pad>': 0, '<unk>': 1}
for word, count in word_counts.items():
    if count > 1: # min frequency
        vocab[word] = len(vocab)

def text_pipeline(x):
    return [vocab.get(word, vocab['<unk>']) for word in tokenize(x)]

MAX_LEN = 50
PAD_IDX = vocab['<pad>']

class EmotionDataset(Dataset):
    def __init__(self, data_split):
        self.texts = data_split['text']
        self.labels = data_split['ekman_label']
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        tokens = text_pipeline(self.texts[idx])
        if len(tokens) < MAX_LEN:
            tokens.extend([PAD_IDX] * (MAX_LEN - len(tokens)))
        else:
            tokens = tokens[:MAX_LEN]
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

train_dataset_seq = EmotionDataset(dataset['train'])
val_dataset_seq = EmotionDataset(dataset['validation'])

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset_seq, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=2, persistent_workers=True)
val_loader = DataLoader(val_dataset_seq, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=2, persistent_workers=True)

print("Downloading GloVe embeddings (this may take a minute)...")
glove_zip = "glove.6B.zip"
glove_txt = "glove.6B.100d.txt"
if not os.path.exists(glove_txt):
    urllib.request.urlretrieve("http://nlp.stanford.edu/data/glove.6B.zip", glove_zip)
    with zipfile.ZipFile(glove_zip, 'r') as zip_ref:
        zip_ref.extract(glove_txt)

glove_embeddings = {}
with open(glove_txt, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        glove_embeddings[word] = vector

embedding_matrix = torch.zeros((len(vocab), 100))
for word, i in vocab.items():
    if word in glove_embeddings:
        embedding_matrix[i] = torch.tensor(glove_embeddings[word])
    else:
        embedding_matrix[i] = torch.randn(100)


In [ ]:
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, classification_report

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_sequence_model(model_name, model, train_loader, val_loader, epochs=30, lr=1e-3, patience=5):
    print(f"\n--- Training {model_name} ---")
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    num_params = count_parameters(model)
    model_size_mb = num_params * 4 / (1024 ** 2)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    model = model.to(device)
    
    best_val_f1 = 0
    best_val_acc = 0
    best_class_report = None
    best_epoch = -1
    epochs_no_improve = 0
    train_start_time = time.time()
    
    history = []
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for texts, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            texts, labels = texts.to(device), labels.to(device)
            optimizer.zero_grad()
            predictions = model(texts)
            loss = criterion(predictions, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for texts, labels in val_loader:
                texts, labels = texts.to(device), labels.to(device)
                predictions = model(texts)
                preds = torch.argmax(predictions, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        val_acc = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
        
        history.append({
            "epoch": epoch + 1,
            "train_loss": total_loss/len(train_loader),
            "val_acc": val_acc,
            "val_f1": val_f1
        })
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_val_acc = val_acc
            best_epoch = epoch + 1
            epochs_no_improve = 0
            best_class_report = classification_report(all_labels, all_preds, target_names=ekman_labels, output_dict=True)
            model_to_save = model.module if hasattr(model, 'module') else model
            torch.save(model_to_save.state_dict(), f"best_{model_name.replace(' ', '')}.pt")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs!")
                break
            
    train_time = time.time() - train_start_time
    
    # Inference Time Test
    model.eval()
    inf_start = time.time()
    with torch.no_grad():
        for i, (texts, _) in enumerate(val_loader):
            if i >= 5: break # just sample 5 batches
            _ = model(texts.to(device))
    inf_time_per_batch_ms = ((time.time() - inf_start) / 5) * 1000
    
    project_metrics.append({
        "Model": model_name,
        "Accuracy": round(best_val_acc, 4),
        "Macro F1": round(best_val_f1, 4),
        "Per-Class Metrics": best_class_report,
        "Training History": history,
        "Best Epoch": best_epoch,
        "Parameters": num_params,
        "Training Time (s)": round(train_time, 2),
        "Inference Time per Batch (ms)": round(inf_time_per_batch_ms, 2),
        "Model Size (MB)": round(model_size_mb, 2)
    })


## 5. LSTM

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embedding.weight.data.copy_(embedding_matrix)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers, bidirectional=bidirectional, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        _, (hidden, _) = self.rnn(embedded)
        if self.rnn.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden = self.dropout(hidden[-1,:,:])
        return self.fc(hidden)

lstm_model = LSTMClassifier(len(vocab), 100, 128, len(ekman_labels), 2, True, 0.3, PAD_IDX)
train_sequence_model("LSTM", lstm_model, train_loader, val_loader, epochs=30)


## 6. GRU

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embedding.weight.data.copy_(embedding_matrix)
        self.rnn = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, bidirectional=bidirectional, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        _, hidden = self.rnn(embedded)
        if self.rnn.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden = self.dropout(hidden[-1,:,:])
        return self.fc(hidden)

gru_model = GRUClassifier(len(vocab), 100, 128, len(ekman_labels), 2, True, 0.3, PAD_IDX)
train_sequence_model("GRU", gru_model, train_loader, val_loader, epochs=30)


## 7. BiLSTM + Attention

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, rnn_outputs):
        attn_weights = torch.softmax(self.attention(rnn_outputs), dim=1)
        context_vector = torch.sum(attn_weights * rnn_outputs, dim=1)
        return context_vector, attn_weights

class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embedding.weight.data.copy_(embedding_matrix)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers, bidirectional=True, dropout=dropout, batch_first=True)
        self.attention = Attention(hidden_dim * 2)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        rnn_outputs, _ = self.rnn(embedded)
        context_vector, attn_weights = self.attention(rnn_outputs)
        return self.fc(self.dropout(context_vector))

bilstm_att_model = BiLSTMAttentionClassifier(len(vocab), 100, 128, len(ekman_labels), 2, 0.3, PAD_IDX)
train_sequence_model("BiLSTM Attention", bilstm_att_model, train_loader, val_loader, epochs=30)


## 8. Transformers (DistilBERT & RoBERTa)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate
from sklearn.metrics import classification_report

def train_transformer(model_name, dataset, num_labels, epochs=3):
    print(f"\n--- Training {model_name} ---")
    hf_tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def tokenize_function(examples):
        return hf_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)
        
    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    
    if "labels" in tokenized_datasets["train"].column_names:
        tokenized_datasets = tokenized_datasets.remove_columns(["labels"])
        
    tokenized_datasets = tokenized_datasets.rename_column("ekman_label", "labels")
    tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    
    num_params = count_parameters(model)
    model_size_mb = num_params * 4 / (1024 ** 2)
    
    if "roberta" in model_name:
        model.config.gradient_checkpointing = True

    metric_acc = evaluate.load("accuracy")
    metric_f1 = evaluate.load("f1")
    
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
        f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
        return {"accuracy": acc, "f1": f1}

    training_args = TrainingArguments(
        output_dir=f"./results_{model_name.replace('/', '_')}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=64, 
        per_device_eval_batch_size=64,
        num_train_epochs=epochs,
        weight_decay=0.01,
        fp16=True, 
        dataloader_num_workers=2,
        dataloader_persistent_workers=True,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    train_start = time.time()
    trainer.train()
    train_time = time.time() - train_start
    
    # Extract History
    history_dict = {}
    for log in trainer.state.log_history:
        if "epoch" not in log: continue
        ep = int(round(log["epoch"]))
        if ep not in history_dict:
            history_dict[ep] = {"epoch": ep}
        if "loss" in log:
            history_dict[ep]["train_loss"] = log["loss"]
        if "eval_f1" in log:
            history_dict[ep]["val_f1"] = log["eval_f1"]
            history_dict[ep]["val_acc"] = log["eval_accuracy"]
            
    history = sorted(list(history_dict.values()), key=lambda x: x["epoch"])
    best_epoch = -1
    best_f1 = 0
    for h in history:
        if "val_f1" in h and h["val_f1"] > best_f1:
            best_f1 = h["val_f1"]
            best_epoch = h["epoch"]
            
    
    output_path = f"./saved_{model_name.replace('/', '_')}"
    trainer.save_model(output_path)
    hf_tokenizer.save_pretrained(output_path)
    
    inf_start = time.time()
    eval_results = trainer.evaluate()
    
    preds_output = trainer.predict(tokenized_datasets["validation"])
    preds = np.argmax(preds_output.predictions, axis=-1)
    class_report = classification_report(preds_output.label_ids, preds, target_names=ekman_labels, output_dict=True)
    
    inf_time_per_batch_ms = ((time.time() - inf_start) / (len(tokenized_datasets["validation"])/64)) * 1000
    
    print(f"Final Val F1 for {model_name}: {eval_results['eval_f1']:.4f}")
    
    project_metrics.append({
        "Model": model_name,
        "Accuracy": round(eval_results['eval_accuracy'], 4),
        "Macro F1": round(eval_results['eval_f1'], 4),
        "Per-Class Metrics": class_report,
        "Training History": history,
        "Best Epoch": best_epoch,
        "Parameters": num_params,
        "Training Time (s)": round(train_time, 2),
        "Inference Time per Batch (ms)": round(inf_time_per_batch_ms, 2),
        "Model Size (MB)": round(model_size_mb, 2)
    })

train_transformer("distilbert-base-uncased", dataset, len(ekman_labels), epochs=15)


In [ ]:
train_transformer("roberta-base", dataset, len(ekman_labels), epochs=15)


## 9. Export & Download

In [ ]:
import shutil
import os
from IPython.display import FileLink, display

KAGGLE_WORK_DIR = "/kaggle/working"
if not os.path.exists(KAGGLE_WORK_DIR):
    KAGGLE_WORK_DIR = "." # Fallback for local testing

# Create a clean export directory
EXPORT_DIR = os.path.join(KAGGLE_WORK_DIR, "emotion_export")
os.makedirs(os.path.join(EXPORT_DIR, "models"), exist_ok=True)

# Save vocab and mappings
mappings = {
    'ekman_labels': ekman_labels,
    'ekman_to_id': ekman_to_id,
    'id_to_ekman': id_to_ekman
}
with open(os.path.join(EXPORT_DIR, "models", "label_mappings.json"), "w") as f:
    json.dump(mappings, f)
    
torch.save(vocab, os.path.join(EXPORT_DIR, "models", "vocab.pth"))

with open(os.path.join(EXPORT_DIR, "models", "metrics.json"), "w") as f:
    json.dump(project_metrics, f, indent=4)

# Move PyTorch model files safely to export
files_to_move = ["best_LSTM.pt", "best_GRU.pt", "best_BiLSTMAttention.pt"]
for file in files_to_move:
    src = os.path.join(KAGGLE_WORK_DIR, file)
    dst = os.path.join(EXPORT_DIR, "models", file)
    if os.path.exists(src):
        shutil.move(src, dst)

# Move Transformer folders safely to export
for item in os.listdir(KAGGLE_WORK_DIR):
    if item.startswith("saved_") and os.path.isdir(os.path.join(KAGGLE_WORK_DIR, item)):
        shutil.move(os.path.join(KAGGLE_WORK_DIR, item), os.path.join(EXPORT_DIR, item))

print("Zipping outputs for download...")
zip_path = shutil.make_archive(os.path.join(KAGGLE_WORK_DIR, 'emotion_classification_outputs'), 'zip', root_dir=EXPORT_DIR)
print(f"Zip created successfully: {zip_path}")

# Smart Direct Download Link
print("Click the link below to download your complete project outputs:")
display(FileLink('emotion_classification_outputs.zip'))
